In [0]:
from pyspark.sql import functions as F

In [0]:
file_path = "abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/raw_data/selected-price-indexes-june-2026.csv"

df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(file_path)
)

df.display()

In [0]:
df.explain("formatted")

In [0]:
df.select(F.spark_partition_id()).distinct().count()

In [0]:
df = df.repartition(10)
df.select(F.spark_partition_id()).distinct().count()

In [0]:
df = df.repartition(1)
df.select(F.spark_partition_id()).distinct().count()

In [0]:
df = df.coalesce(1)
df.select(F.spark_partition_id()).distinct().count()

In [0]:
(
    df.write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/sales2")
    .saveAsTable("qc_databricks_batch_10.qc_db.price_index2")
)

In [0]:
cus_df = spark.table("samples.bakehouse.sales_customers")
trans_df = spark.table("samples.bakehouse.sales_transactions")

In [0]:
brod_result_df = (
    cus_df
    .join(
        F.broadcast(trans_df), 
        on=["customerID"],
        how="inner"
    )
)
brod_result_df.display()

In [0]:
(
    brod_result_df.write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/customer_transactions")
    .saveAsTable("qc_databricks_batch_10.qc_db.customer_transactions")
)